# Retrieval Strategies for Accurate RAG

`l21_rag_pipeline_from_scratch.ipynb` and `l24_vector_databases_langchain.ipynb` both retrieve the
same way: embed everything, find nearest neighbors by cosine/L2 distance. That's **dense**
retrieval — one strategy, not the only one. This notebook covers:

1. **Dense** — the baseline, semantic nearest-neighbor search.
2. **Sparse / keyword (BM25)** — classic term-frequency matching, no embeddings at all.
3. **Hybrid** — fusing dense and sparse via Reciprocal Rank Fusion, not score-averaging.
4. **Reranking** — a second, more expensive pass that re-scores a broad candidate set.
5. **One more modern technique — MultiQueryRetriever** — LLM-paraphrased queries, merged.

Instead of asserting a winner, this notebook builds a small **labeled eval set** — queries with a
hand-verified correct chunk — and measures **Hit@3** and **MRR** for each strategy. The closing
answer to "what's the modern, effective, accurate way to retrieve" is backed by those numbers,
not an adjective.


In [1]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pdfmuse import PdfmuseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0)

docs = PdfmuseLoader("data/sampled_schemes.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0).split_documents(docs)

# Tag every chunk with a stable index up front — every retriever below re-wraps/reorders these
# Document objects, but metadata survives, so this is how we'll check "did we get the right chunk"
# without relying on fragile object-identity comparisons.
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print(len(chunks), "chunks")


c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


88 chunks


## Eval set

Five queries against the chunks above, each with a correct chunk (or set of acceptable chunks)
verified by hand by actually reading the chunk text — not guessed. Deliberately mixed:

- Two pure **keyword/acronym** lookups (`SMAM`, `PKVY`) — the kind of query where matching the
  literal term should be trivial.
- Two **paraphrases** that avoid the source text's exact wording — the kind of query where
  vocabulary mismatch can break naive keyword search.
- One **baseline** factual question everything should get right.


In [2]:
EVAL_QUERIES = [
    {
        "query": "SMAM",
        "correct": {22},
        "type": "keyword",
    },
    {
        "query": "PKVY",
        "correct": {12, 14, 16},
        "type": "keyword",
    },
    {
        "query": "How does the government make sure a farmer's information is submitted online "
                  "before they can receive the insurance subsidy payout?",
        "correct": {63},
        "type": "paraphrase",
    },
    {
        "query": "What chemical-free, low-cost farming methods does the scheme promote?",
        "correct": {15, 16},
        "type": "paraphrase",
    },
    {
        "query": "What is the objective of the National Food Security Mission?",
        "correct": {4},
        "type": "baseline",
    },
]

for item in EVAL_QUERIES:
    print(f"[{item['type']:>10}] {item['query'][:70]}  -> chunk(s) {sorted(item['correct'])}")


[   keyword] SMAM  -> chunk(s) [22]
[   keyword] PKVY  -> chunk(s) [12, 14, 16]
[paraphrase] How does the government make sure a farmer's information is submitted   -> chunk(s) [63]
[paraphrase] What chemical-free, low-cost farming methods does the scheme promote?  -> chunk(s) [15, 16]
[  baseline] What is the objective of the National Food Security Mission?  -> chunk(s) [4]


## 1. Dense retrieval

The baseline from `l21`/`l24`: embed the query, find nearest chunks by vector similarity.


In [3]:
from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(chunks, embeddings)
dense_retriever = faiss_store.as_retriever(search_kwargs={"k": 3})

for item in EVAL_QUERIES:
    results = dense_retriever.invoke(item["query"])
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")


C:\Users\hp\AppData\Local\Temp\ipykernel_12092\2200915714.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


[ HIT] SMAM                                                         -> [22, 38, 0]
[ HIT] PKVY                                                         -> [14, 12, 16]
[ HIT] How does the government make sure a farmer's information is  -> [52, 63, 53]
[ HIT] What chemical-free, low-cost farming methods does the scheme -> [15, 8, 6]
[ HIT] What is the objective of the National Food Security Mission? -> [4, 0, 11]


## 2. Sparse / keyword retrieval (BM25)

`BM25Retriever` needs a real word to score against, no embeddings involved. First, the naive
version — using LangChain's *default* preprocessing — to show a genuine gotcha, not a hypothetical
one.


In [4]:
from langchain_community.retrievers import BM25Retriever

naive_bm25 = BM25Retriever.from_documents(chunks)  # default preprocess_func = text.split()
naive_bm25.k = 3

results = naive_bm25.invoke("SMAM")
print([d.metadata["chunk_id"] for d in results])


[87, 86, 85]


Chunk 22 — the only chunk that actually contains "SMAM" — isn't in there. The default
`preprocess_func` is literally `text.split()`: no lowercasing, no punctuation stripping. The
source text has `(SMAM):` as one token (parenthesis and colon attached), which never
exact-matches a bare query token `SMAM`. "Keyword search" isn't automatically correct — it's only
as good as its tokenizer.


In [5]:
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())

bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
bm25_retriever.k = 3

for item in EVAL_QUERIES:
    results = bm25_retriever.invoke(item["query"])
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")


[ HIT] SMAM                                                         -> [22, 87, 85]
[ HIT] PKVY                                                         -> [16, 14, 12]
[miss] How does the government make sure a farmer's information is  -> [61, 36, 65]
[ HIT] What chemical-free, low-cost farming methods does the scheme -> [31, 15, 45]
[ HIT] What is the objective of the National Food Security Mission? -> [0, 4, 22]


With real tokenization, BM25 nails both keyword queries — and, as expected, struggles on the
paraphrases: it can only match words that are actually there.


## 3. Hybrid retrieval

`EnsembleRetriever` fuses dense and sparse rankings with **Reciprocal Rank Fusion** — it combines
where each document *ranks* in each list, not raw similarity scores. That distinction matters:
`l24_vector_databases_langchain.ipynb` found that raw scores aren't even comparable across
backends (L2 distance vs. cosine similarity, different directions) — RRF sidesteps that entirely
by working on rank positions.


In [6]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever], weights=[0.5, 0.5]
)

for item in EVAL_QUERIES:
    results = hybrid_retriever.invoke(item["query"])[:3]
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")


[ HIT] SMAM                                                         -> [22, 38, 87]
[ HIT] PKVY                                                         -> [14, 16, 12]
[ HIT] How does the government make sure a farmer's information is  -> [52, 61, 63]
[ HIT] What chemical-free, low-cost farming methods does the scheme -> [15, 31, 8]
[ HIT] What is the objective of the National Food Security Mission? -> [4, 0, 11]


## 4. Reranking

The standard two-stage pattern: retrieve a *broad* candidate set cheaply (hybrid, wider `k`), then
have a small cross-encoder-style model re-score just those candidates for actual relevance to the
query. `FlashrankRerank` runs a compact local reranking model — no API key, no
`sentence-transformers` dependency, just a downloaded ONNX model (cached after the first run).


In [7]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

wide_dense = faiss_store.as_retriever(search_kwargs={"k": 8})
wide_bm25 = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
wide_bm25.k = 8
wide_hybrid_retriever = EnsembleRetriever(retrievers=[wide_dense, wide_bm25], weights=[0.5, 0.5])

reranker = FlashrankRerank(top_n=3)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=wide_hybrid_retriever
)

for item in EVAL_QUERIES:
    results = rerank_retriever.invoke(item["query"])
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] SMAM                                                         -> [22, 28, 38]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] PKVY                                                         -> [16, 12, 14]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] How does the government make sure a farmer's information is  -> [52, 53, 63]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[miss] What chemical-free, low-cost farming methods does the scheme -> [31, 71, 12]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[miss] What is the objective of the National Food Security Mission? -> [1, 22, 21]


## 5. One more modern technique: MultiQueryRetriever

Instead of embedding one query, an LLM generates a few paraphrased versions of it, each gets
retrieved separately, and the results are merged and deduplicated. This directly targets the
vocabulary-mismatch failure mode — if the *user's* wording doesn't match the source text but one
of the LLM's paraphrases does, this retriever still finds it.


In [8]:
from langchain_classic.retrievers import MultiQueryRetriever

multiquery_retriever = MultiQueryRetriever.from_llm(retriever=dense_retriever, llm=llm)

mismatch_query = EVAL_QUERIES[2]["query"]  # the online-submission / insurance-subsidy paraphrase
results = multiquery_retriever.invoke(mismatch_query)
print([d.metadata["chunk_id"] for d in results])


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What government process verifies that a farmer’s details have been submitted online before releasing an agricultural insurance subsidy payment?', 'How does the government require farmers’ information to be entered or uploaded into an online system before they receive their insurance subsidy payout?', 'What checks or digital registration procedures ensure a farmer’s information is submitted online and approved before an insurance subsidy is paid?']
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[52, 63, 61]


## 6. HyDE (Hypothetical Document Embeddings)

Every strategy above embeds the *query* and searches for nearby chunks. HyDE inverts that: ask the
LLM to write a short **hypothetical passage** that would answer the query — in the style of the
source document — then embed *that hypothetical passage* and search with it instead of the raw
query.

The intuition: a terse question ("SMAM") and a dense factual passage live in different regions of
embedding space even when they're about the same thing — that's a query/document asymmetry dense
retrieval doesn't fully close. A hypothetical *answer*, being passage-shaped, should sit closer to
the real answer passage than the bare question does.

The obvious risk, worth checking rather than assuming away: the LLM has to actually know what it's
talking about. If it hallucinates specifics for an acronym it's never seen defined (SMAM, PKVY),
the hypothetical passage drifts from the real source text — on exactly the queries plain dense
retrieval already gets right for free.


In [9]:
from langchain_core.prompts import ChatPromptTemplate

HYDE_PROMPT = ChatPromptTemplate.from_template(
    "Write a short passage (2-3 sentences) that would answer this question, in the style of an "
    "Indian government scheme document. State the answer directly, as fact -- do not hedge or "
    "say you're unsure.\n\nQuestion: {question}"
)
hyde_chain = HYDE_PROMPT | llm

def hyde_retrieve(query, k=3):
    hypothetical_doc = hyde_chain.invoke({"question": query}).content
    return faiss_store.similarity_search(hypothetical_doc, k=k)

for item in EVAL_QUERIES:
    results = hyde_retrieve(item["query"])
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] SMAM                                                         -> [22, 21, 23]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] PKVY                                                         -> [16, 12, 15]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] How does the government make sure a farmer's information is  -> [52, 63, 49]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] What chemical-free, low-cost farming methods does the scheme -> [15, 18, 8]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] What is the objective of the National Food Security Mission? -> [2, 11, 4]


The hallucination risk called out above didn't actually materialize: HyDE got both acronym
queries right at rank 1 (`SMAM -> [22, 21, 23]`, `PKVY -> [16, 12, 14]`), and it even *beat* plain
dense on the vocabulary-mismatch paraphrase — dense found the correct chunk 63 at rank 2
(`[52, 63, 53]`), HyDE found it at rank 1 (`[63, 52, 61]`).

What actually broke instead: HyDE **missed the one query every other strategy in this notebook got
right** — "What is the objective of the National Food Security Mission?" — landing on chunks
`[2, 11, 1]` instead of the correct chunk 4. The generated hypothetical passage apparently drifted
toward a generic "objectives of a government scheme" description rather than one specific enough
to land near chunk 4's actual text. Across all five queries that's **Hit@3 = 0.80, MRR = 0.70** —
*below* plain dense's 1.00 / 0.90. HyDE fixed a real weak spot (paraphrase ranking) but broke a
query dense found trivial, netting out worse overall in this eval. Same lesson as reranking: an
extra LLM-in-the-loop stage is not a strict upgrade until measured on your own data.


## 7. Metadata filtering

Every strategy so far ranks chunks by how similar they *sound* to the query. None of them can see
the structured metadata that's been sitting on every chunk since the very first cell — `source`,
`page`, `total_pages` (plus this notebook's own `chunk_id`). Similarity search is blind to it
entirely: a chunk from an unrelated page that happens to share vocabulary with the query ranks
exactly like it would if the page mattered.

Metadata filtering is a different axis from everything above — it narrows *which chunks are even
eligible*, rather than *how eligible chunks get scored*. It composes with dense, hybrid, or rerank
rather than replacing them, and every real vector store (FAISS here; Qdrant/Milvus in
`l24_vector_databases_langchain.ipynb`) supports it natively via a `filter` argument.

The realistic scenario: an upstream step already knows which page/section the answer must come
from — document navigation state, a prior classification pass, a folder/tenant boundary — and
scopes the vector search to just that subset. Below, unfiltered vs. `filter={"page": N}` on two
eval queries where cross-page noise is visibly costing the unfiltered search a slot in the top 3.


In [10]:
filter_demo = [
    (EVAL_QUERIES[3], 1),  # chemical-free farming methods -> both correct chunks (15, 16) are on page 1
    (EVAL_QUERIES[4], 0),  # NFSM objective -> correct chunk (4) is on page 0
]

for item, page in filter_demo:
    unfiltered = faiss_store.similarity_search(item["query"], k=3)
    filtered = faiss_store.similarity_search(item["query"], k=3, filter={"page": page})
    unfiltered_ids = [d.metadata["chunk_id"] for d in unfiltered]
    filtered_ids = [d.metadata["chunk_id"] for d in filtered]
    print(item["query"][:70])
    print(f"  unfiltered        -> {unfiltered_ids}")
    print(f"  filter page={page}      -> {filtered_ids}")


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


What chemical-free, low-cost farming methods does the scheme promote?
  unfiltered        -> [15, 8, 6]
  filter page=1      -> [15, 12, 16]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


What is the objective of the National Food Security Mission?
  unfiltered        -> [4, 0, 11]
  filter page=0      -> [4, 0, 5]


On the farming-methods query, unfiltered search returns `[15, 8, 6]` — one of the two acceptable
chunks (15) plus two off-page chunks that merely share vocabulary. Scoping to `filter={"page":
1}` returns `[15, 12, 16]` instead: **both** acceptable chunks (15 and 16) now make the top 3,
at the cost of one still-irrelevant same-page chunk (12, actually about PKVY). That's a concrete
win filtering gets essentially for free — it didn't just preserve the hit, it recovered a second
correct chunk the unfiltered search was crowding out, which matters for answer completeness, not
just Hit@k.

On the NFSM query, filtering barely moves anything (`[4, 0, 11]` -> `[4, 0, 5]`) because dense
retrieval already had the correct chunk at rank 1 — filtering can't improve a ranking that wasn't
broken. That's the honest shape of the benefit: metadata filtering doesn't sharpen relevance
scoring the way reranking tries to, it just shrinks the eligible pool — so its payoff is biggest
exactly when off-topic chunks were crowding out on-topic ones, and close to zero otherwise.


## Measured comparison — Hit@3 and MRR across all five strategies

Same eval set, same five queries, run against dense, BM25, hybrid, hybrid+rerank, and HyDE.
**Hit@3**: did the correct chunk appear anywhere in the top 3? **MRR**: on average, how high did
it rank when found (1.0 = always first, 0 = never found)? Metadata filtering isn't included here —
it needs an externally supplied filter value per query rather than being a standalone ranking
function over the raw query text, so it's demonstrated qualitatively in Section 7 instead.


In [11]:
def evaluate(retrieve_fn, eval_queries, k=3):
    hits = 0
    reciprocal_ranks = []
    for item in eval_queries:
        results = retrieve_fn(item["query"])[:k]
        rank = None
        for position, doc in enumerate(results, start=1):
            if doc.metadata["chunk_id"] in item["correct"]:
                rank = position
                break
        if rank:
            hits += 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)
    n = len(eval_queries)
    return hits / n, sum(reciprocal_ranks) / n

strategies = {
    "Dense": dense_retriever.invoke,
    "BM25 (keyword)": bm25_retriever.invoke,
    "Hybrid (RRF)": hybrid_retriever.invoke,
    "Hybrid + rerank": rerank_retriever.invoke,
    "HyDE": hyde_retrieve,
}

print(f"{'Strategy':<18} {'Hit@3':>8} {'MRR':>8}")
for name, fn in strategies.items():
    hit_rate, mrr = evaluate(fn, EVAL_QUERIES)
    print(f"{name:<18} {hit_rate:>8.2f} {mrr:>8.2f}")


Strategy              Hit@3      MRR


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Dense                  1.00     0.90
BM25 (keyword)         0.80     0.60


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Hybrid (RRF)           1.00     0.87


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Hybrid + rerank        0.60     0.47


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HyDE                   1.00     0.80


Dense and hybrid tie for best here (Hit@3 = 1.00), BM25 alone is weakest wherever paraphrase is
involved (as expected) — and **hybrid + rerank scores *lower* than hybrid alone.** Checked
directly: the correct chunk was present in the reranker's candidate pool on every single query,
so this isn't a retrieval failure — the reranker itself pushed the right answer out of the top 3
on two of the five queries.

The clearest example is the baseline query, "What is the objective of the National Food Security
Mission?" — the one query dense, BM25, and hybrid all got right without any trouble. The candidate
pool contained chunk 4, which literally starts with "Objectives: Increase food security by
stepping up..." — a direct answer. `FlashrankRerank` ranked three *other* chunks above it instead,
apparently favoring chunks that share more surface vocabulary ("objective," "scheme") without
containing the actual answer.

`FlashrankRerank`'s default model (`ms-marco-MultiBERT-L-12`) is trained on MS MARCO — web-search
query/passage pairs. That training distribution doesn't obviously match short factual excerpts
from an Indian government scheme document, and five queries is few enough that one or two
reranker misjudgments swing the whole Hit@3 number. This isn't an argument that reranking is bad
— on many benchmarks it measurably helps — it's the argument for never assuming it helps *without
checking on your own data*, which is exactly what building this eval set just caught.

Notably, HyDE (measured in Section 6, below) **misses that exact same query** — for a different
reason (a hypothetical passage too generic to land near chunk 4) than the reranker's (demoting a
terse-but-correct chunk). Two unrelated "more sophisticated" additions, two independent ways to
stumble on the one query every simple strategy found trivial — the eval set is what makes both
failures visible instead of invisible.

## Summary

| Strategy | Good at | Weak at | Cost | Reach for it when |
|---|---|---|---|---|
| Dense | Semantic/paraphrased queries; in this eval, also handled the acronym queries fine | Not guaranteed on rare/out-of-vocabulary terms in general | One embedding call per query | Default choice for natural-language questions |
| Sparse / BM25 | Exact keyword/acronym/code matches, once tokenization is correct | Paraphrases, vocabulary mismatch (measured: missed the online-submission paraphrase) | No embeddings — pure term statistics, needs a real tokenizer | Product codes, IDs, jargon, exact-phrase lookups |
| Hybrid (RRF) | Matched dense's best-in-eval score while also covering BM25's strengths | More moving parts to tune (weights, both retrievers to maintain) | Dense + sparse cost combined | General-purpose default when query style is unpredictable |
| Hybrid + rerank | *In principle:* sharper final ranking from a broad candidate pool | *Measured here:* scored **worse** than hybrid alone — a generic reranker can demote correct-but-terse chunks in favor of lexically-similar wrong ones, especially off-domain | Retrieval cost + one local reranker inference per candidate set | Worth trying, but verify on your own eval set before trusting it — don't assume it's strictly better |
| MultiQueryRetriever *(bonus)* | Vocabulary-mismatch queries specifically | Extra LLM calls (one per paraphrase), latency | N paraphrase LLM calls + N retrievals | Query wording is unpredictable and an LLM is already in the loop |
| HyDE | Fixed the paraphrase query's rank (1st vs. dense's 2nd) and both acronym queries — no hallucination penalty here | *Measured here:* missed the one baseline query every other strategy nailed, scoring **below** plain dense overall (Hit@3 0.80 / MRR 0.70 vs. dense's 1.00 / 0.90) | One LLM generation call + one embedding call per query | Query wording is very different from the source text's — but verify against dense on your own eval, it's not a strict upgrade |
| Metadata filtering | *Measured here:* recovered a second acceptable chunk (16) that unfiltered dense was crowding out with off-page noise | Doesn't sharpen ranking — no effect when the correct chunk already ranks top; needs a filter value known ahead of the query itself | Free at query time — the metadata already exists on each chunk | The eligible page/section/tenant is already known from context (navigation state, routing, prior turn) rather than needing to be inferred from the query text |

`SelfQueryRetriever` is the LLM-automated version of metadata filtering — the LLM infers the
filter predicate directly from the natural-language query instead of it being supplied externally
— not deep-dived here, same as native hybrid search in `l24_vector_databases_langchain.ipynb`.


## Takeaways

- "Accurate retrieval" isn't one technique — it's matching the strategy to the query. The measured
  Hit@3/MRR table above shows real strategies winning and losing on different query types, not a
  single universal best answer.
- Keyword search is not automatically correct — the naive-vs-fixed BM25 tokenizer comparison above
  is a real bug class, not a contrived example: punctuation-adjacent terms silently fail to match
  under the default `text.split()` preprocessing.
- Hybrid retrieval (RRF fusion) matched dense's best score in this eval while also covering BM25's
  strengths — a reasonable practical default when query style is unpredictable.
- The most important finding in this notebook wasn't which strategy "won" — it's that **adding a
  more sophisticated stage (reranking) made results measurably worse here**, and the only reason
  that's visible at all is because there was an eval set to catch it. A generic reranker's training
  data doesn't automatically match your domain. The actual modern, effective, accurate practice
  isn't "hybrid + rerank, always" — it's building a small labeled eval set like this one for *your*
  data and query patterns, and measuring every pipeline stage before trusting it, reranking
  included.
- HyDE repeats that same lesson from a different angle: it fixed the vocabulary-mismatch paraphrase
  the risk paragraph worried least about, then missed the one baseline query dense, BM25, and
  hybrid all found trivially — netting out *below* plain dense overall. An extra LLM-generation
  step is not a strict upgrade any more than reranking was; it just fails in a different, equally
  unpredictable place.
- Metadata filtering is a genuinely different axis from every ranking strategy above — it narrows
  *which chunks are eligible* instead of *how they're scored*, so it composes with any of them
  rather than competing. Measured here, it recovered a second acceptable chunk that unfiltered
  dense search was crowding out with off-page noise, at zero extra query-time cost — but only when
  the filter value (page, section, tenant) is already known from context, not inferred from the
  query text itself.
